<a href="https://colab.research.google.com/github/jasswinder8003-oss/fake-news-detection-nlp-ml/blob/main/Document_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import re
import string
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], ignore_index=True)

print("Dataset shape:", df.shape)
print(df.head())

In [ ]:
df = df[["title", "text", "label"]].copy()

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = df["title"] + " " + df["text"]

df = df[["content", "label"]]

print(df.shape)
print(df["label"].value_counts())

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("\nDataset after removing duplicates:", df.shape)

In [ ]:
print(df["label"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x="label", data=df)
plt.xticks([0, 1], ["Fake", "Real"])
plt.xlabel("News Category")
plt.ylabel("Number of Articles")
plt.title("Distribution of Fake and Real News")
plt.show()

In [ ]:
df["text_length"] = df["content"].str.len()

print(df["text_length"].describe())

plt.figure(figsize=(8, 4))
plt.hist(df["text_length"], bins=50)
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.title("Distribution of News Text Length")
plt.show()

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [ ]:
df["clean_text"] = df["content"].apply(clean_text)

df[["content", "clean_text", "label"]].head()

In [ ]:
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Final dataset size:", df.shape)

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_tfidf, y_train)

logistic_pred = logistic_model.predict(X_test_tfidf)

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

In [ ]:
svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

In [ ]:
models = {
    "Logistic Regression": logistic_pred,
    "Naive Bayes": nb_pred,
    "Linear SVM": svm_pred
}

results = []

for model_name, predictions in models.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
results_df.set_index("Model").plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Comparison of Machine Learning Models")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

In [ ]:
print("Logistic Regression")
print(classification_report(
    y_test,
    logistic_pred,
    target_names=["Fake", "Real"]
))

In [ ]:
print("Naive Bayes")
print(classification_report(
    y_test,
    nb_pred,
    target_names=["Fake", "Real"]
))

In [ ]:
print("Linear SVM")
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Fake", "Real"]
))

In [ ]:
predictions = {
    "Logistic Regression": logistic_pred,
    "Naive Bayes": nb_pred,
    "Linear SVM": svm_pred
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, prediction) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, prediction)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Fake", "Real"],
        yticklabels=["Fake", "Real"],
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.loc[
    results_df["F1-Score"].idxmax(), "Model"
]

print("Best-performing model:", best_model_name)

print(
    results_df[
        results_df["Model"] == best_model_name
    ]
)

In [ ]:
trained_models = {
    "Logistic Regression": logistic_model,
    "Naive Bayes": nb_model,
    "Linear SVM": svm_model
}

best_model = trained_models[best_model_name]

In [ ]:
def predict_news(news_text):
    cleaned = clean_text(news_text)
    features = tfidf.transform([cleaned])
    prediction = best_model.predict(features)[0]

    if prediction == 0:
        return "FAKE NEWS"
    else:
        return "REAL NEWS"

In [ ]:
news = input("Enter news text: ")

result = predict_news(news)

print("Prediction:", result)